# Data Preprocessing for MBIC (BABE), SG1, SG2, and cc_news
This notebook ensures consistent `article_id`s across all datasets so that `build_graph.py` can correctly match annotations and compute agreements. It then creates a single, clean CSV for the embedding pipeline, containing ONLY the useful columns.

In [6]:
import pandas as pd
from datasets import load_dataset
import uuid
import hashlib

def get_id(text):
    # Generates a consistent ID based on the text if no ID exists
    return hashlib.md5(str(text).strip().encode('utf-8')).hexdigest()

In [7]:
print("Processing MBIC, SG1, and SG2 to ensure consistent article_ids...")

# Load datasets
df_babe = pd.read_csv('data/final_labels_MBIC.csv', sep=';', on_bad_lines='skip')
df_sg1 = pd.read_csv('data/raw_labels_SG1.csv', sep=';', on_bad_lines='skip')
df_sg2 = pd.read_csv('data/raw_labels_SG2.csv', sep=';', on_bad_lines='skip')

# Safely add consistent article_id based on text if they don't already have one!
if 'article_id' not in df_babe.columns:
    df_babe['article_id'] = df_babe['text'].apply(get_id)
    df_babe.to_csv('data/final_labels_MBIC.csv', sep=';', index=False)

if 'article_id' not in df_sg1.columns:
    df_sg1['article_id'] = df_sg1['text'].apply(get_id)
    df_sg1.to_csv('data/raw_labels_SG1.csv', sep=';', index=False)

if 'article_id' not in df_sg2.columns:
    df_sg2['article_id'] = df_sg2['text'].apply(get_id)
    df_sg2.to_csv('data/raw_labels_SG2.csv', sep=';', index=False)

print("Consistent article_ids applied and saved! Your original files remain fully intact for compute_agreement().")

Processing MBIC, SG1, and SG2 to ensure consistent article_ids...
Consistent article_ids applied and saved! Your original files remain fully intact for compute_agreement().


In [8]:
print("Extracting clean data from MBIC, SG1, and SG2 for embeddings...")

# Combine them all but WE DROP ALL USELESS COLUMNS here. We ONLY keep article_id, text, and label_bias.
all_annotated = pd.concat([df_babe, df_sg1, df_sg2], ignore_index=True)
unique_articles = all_annotated[['article_id', 'text', 'label_bias']].drop_duplicates(subset=['article_id'])
unique_articles = unique_articles.rename(columns={'text': 'body'})

print(f"Found {len(unique_articles)} unique annotated articles.")

Extracting clean data from MBIC, SG1, and SG2 for embeddings...
Found 3698 unique annotated articles.


In [9]:
print("Loading unlabeled cc_news...")
cc_news_dataset = load_dataset('cc_news', split='train').shuffle(seed=42)
cc_news_subset = cc_news_dataset.select(range(10000)) # Change this to add more unlabeled data

df_cc = pd.DataFrame(cc_news_subset)[['text']].rename(columns={'text': 'body'})

# Generate random UUIDs for cc_news since they don't need to match any annotations
df_cc['article_id'] = [str(uuid.uuid4()) for _ in range(len(df_cc))]
# cc_news has no labels, so we explicitly mark them as Unlabeled
df_cc['label_bias'] = 'Unlabeled'

print(f"Added {len(df_cc)} unlabeled articles from cc_news.")

Loading unlabeled cc_news...
Added 10000 unlabeled articles from cc_news.


In [10]:
print("Merging all data for the embedding pipeline...")
final_df = pd.concat([unique_articles, df_cc], ignore_index=True)
final_df = final_df.dropna(subset=['body'])
final_df['body'] = final_df['body'].astype(str).str.strip()
final_df = final_df[final_df['body'] != '']

# The final CSV will now have exactly 3 columns: article_id, body, label_bias. ALL useless columns are completely gone.
output_csv = 'data/merged_clean_data.csv'
final_df.to_csv(output_csv, index=False)
print(f"Saved {len(final_df)} total articles to {output_csv}.")
print("You can now run article_embeddings.py on this file!")
final_df.head()

Merging all data for the embedding pipeline...
Saved 13697 total articles to data/merged_clean_data.csv.
You can now run article_embeddings.py on this file!


,article_id,body,label_bias
0,55ab5220d9a45042935cd79d7af9a0bc,YouTube is making clear there will be no “birt...,Biased
1,bccd148dab0e838f9752d30b41bed0f8,So while there may be a humanitarian crisis dr...,Biased
2,f1ed0406165d1c41e0c63b8f5ba0d787,"Looking around the United States, there is nev...",Biased
3,d40cf84bb7fb044f1032894197c43ba6,The Republican president assumed he was helpin...,Biased
4,ae0bfa8bbb710f70c153cd536fe629f5,The explosion of the Hispanic population has l...,Biased
